### Gold layer: team_ratings

Goal: Dixon-Coles attack/defense ratings per team, fitted point-in-time
(each matchday uses only matches played before it).

Fitted separately for each league. Read-only until the diagnostics are done.

In [1]:
import sys
import time
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path.cwd()
sys.path.append(str(PROJECT_ROOT / "src"))

import pyarrow as pa
from utils.parquet_io import read_league_all_seasons, read_partition, write_partition
from utils.dixon_coles import fit_dixon_coles
from utils.schema import TEAM_RATINGS_SCHEMA

LAKE_ROOT = PROJECT_ROOT / "data"
LEAGUES = ["laliga", "primeira_liga"]

print("Lake root exists:", LAKE_ROOT.exists())

Lake root exists: True


In [2]:
silver = {}
for league in LEAGUES:
    silver[league] = read_league_all_seasons(
        LAKE_ROOT, layer="silver", league=league
    ).to_pandas()
    print(f"{league}: {silver[league].shape}")

print()
print(silver["laliga"].dtypes)
silver["laliga"].head(3).T

laliga: (1520, 19)
primeira_liga: (1224, 19)

league                          object
season                          object
matchday                         int32
match_date         datetime64[ms, UTC]
home_team                       object
away_team                       object
home_team_id                    object
away_team_id                    object
home_score                     float64
away_score                     float64
status                          object
home_xg                        float32
away_xg                        float32
source                          object
home_team_key                   object
away_team_key                   object
result                          object
goal_difference                  Int32
_ingested_at                    object
dtype: object


,0,1,2
league,laliga,laliga,laliga
season,2023-2024,2023-2024,2023-2024
matchday,1,1,1
match_date,2023-08-11 17:30:00+00:00,2023-08-11 20:00:00+00:00,2023-08-12 15:00:00+00:00
home_team,UD Almería,Sevilla FC,Real Sociedad de Fútbol
away_team,Rayo Vallecano de Madrid,Valencia CF,Girona FC
home_team_id,267,559,92
away_team_id,87,95,298
home_score,0.0,1.0,1.0
away_score,2.0,2.0,1.0


In [3]:
for league, df in silver.items():
    print(f"=== {league} ===")
    print(df.groupby("season")["status"].value_counts().unstack(fill_value=0))
    print("xG non-null:", df.groupby("season")["home_xg"].apply(lambda s: int(s.notna().sum())).to_dict())
    print("Score non-null:", df.groupby("season")["home_score"].apply(lambda s: int(s.notna().sum())).to_dict())
    print("Max matchday:", df.groupby("season")["matchday"].max().to_dict())
    print()

=== laliga ===
status     FINISHED  SCHEDULED  TIMED
season                               
2023-2024       380          0      0
2024-2025       380          0      0
2025-2026       380          0      0
2026-2027        69        280     31
xG non-null: {'2023-2024': 0, '2024-2025': 0, '2025-2026': 0, '2026-2027': 0}
Score non-null: {'2023-2024': 380, '2024-2025': 380, '2025-2026': 380, '2026-2027': 70}
Max matchday: {'2023-2024': 38, '2024-2025': 38, '2025-2026': 38, '2026-2027': 38}

=== primeira_liga ===
status     FINISHED  POSTPONED  SCHEDULED  TIMED
season                                          
2023-2024       306          0          0      0
2024-2025       306          0          0      0
2025-2026       306          0          0      0
2026-2027        62          1        198     45
xG non-null: {'2023-2024': 0, '2024-2025': 0, '2025-2026': 0, '2026-2027': 0}
Score non-null: {'2023-2024': 306, '2024-2025': 306, '2025-2026': 306, '2026-2027': 62}
Max matchday: {'2023-2024

In [4]:
for league, df in silver.items():
    seasons = sorted(df["season"].unique())
    teams = {s: set(df.loc[df["season"] == s, "home_team_key"]) for s in seasons}
    print(f"=== {league} ===")
    for prev, cur in zip(seasons, seasons[1:]):
        print(cur, "new teams:", sorted(teams[cur] - teams[prev]))
    print()

=== laliga ===
2024-2025 new teams: ['cd_leganés', 'rcd_espanyol_de_barcelona', 'real_valladolid_cf']
2025-2026 new teams: ['elche_cf', 'levante_ud', 'real_oviedo']
2026-2027 new teams: ['málaga_cf', 'rc_deportivo', 'real_racing_club']

=== primeira_liga ===
2024-2025 new teams: ['avs', 'cd_nacional', 'cd_santa_clara']
2025-2026 new teams: ['cd_tondela', 'fc_alverca']
2026-2027 new teams: ['académico_de_viseu', 'marítimo']



In [5]:
import importlib
p = PROJECT_ROOT / "src" / "utils" / "dixon_coles.py"
print("Exists:", p.exists())
importlib.invalidate_caches()

Exists: True


In [6]:
import time

league = "laliga"
df = silver[league]
teams = sorted(set(df["home_team_key"]) | set(df["away_team_key"]))
played = df[(df["status"] == "FINISHED") & df["home_score"].notna()]
ref_date = played["match_date"].max() + pd.Timedelta(days=1)

t0 = time.time()
ratings, params, res = fit_dixon_coles(played, teams, ref_date)
print(f"Fit time: {time.time()-t0:.1f}s | converged: {res.success}")
print({k: round(v, 3) for k, v in params.items()})
print("\nTop attack:");    print(ratings.nlargest(5, "attack").round(3))
print("\nTop defense:");   print(ratings.nlargest(5, "defense").round(3))
print("\nWorst defense:"); print(ratings.nsmallest(5, "defense").round(3))

Fit time: 0.2s | converged: True
{'mu': 0.119, 'gamma': 0.287, 'rho': -0.013}

Top attack:
                   team_key  attack  defense  n_matches
7              fc_barcelona   0.702    0.259        121
19           real_madrid_cf   0.475    0.313        121
28            villarreal_cf   0.355    0.013        121
3   club_atlético_de_madrid   0.307    0.227        121
18      real_betis_balompié   0.151    0.120        121

Top defense:
                   team_key  attack  defense  n_matches
19           real_madrid_cf   0.475    0.313        121
7              fc_barcelona   0.702    0.259        121
8                 getafe_cf  -0.338    0.251        121
3   club_atlético_de_madrid   0.307    0.227        121
0             athletic_club   0.003    0.152        120

Worst defense:
              team_key  attack  defense  n_matches
21    real_racing_club   0.131   -0.425          7
23  real_valladolid_cf  -0.281   -0.381         38
10          granada_cf  -0.095   -0.200         38
25 

In [7]:
rows = []
for league, df in silver.items():
    seasons = sorted(df["season"].unique())
    for prev, cur in zip(seasons, seasons[1:]):
        if cur == seasons[-1]:
            continue  # current season still in progress
        new = set(df.loc[df["season"] == cur, "home_team_key"]) - set(df.loc[df["season"] == prev, "home_team_key"])
        upto = df[(df["season"] <= cur) & (df["status"] == "FINISHED")]
        teams = sorted(set(upto["home_team_key"]) | set(upto["away_team_key"]))
        ref = upto["match_date"].max() + pd.Timedelta(days=1)
        r, _, _ = fit_dixon_coles(upto, teams, ref)
        rows.append(r[r["team_key"].isin(new)].assign(league=league, season=cur))

promoted = pd.concat(rows)
print(promoted.round(3).to_string(index=False))
print("\nMean promoted rating:")
print(promoted[["attack", "defense"]].mean().round(3))

                 team_key  attack  defense  n_matches        league    season
               cd_leganés  -0.118   -0.078         38        laliga 2024-2025
rcd_espanyol_de_barcelona  -0.107    0.021         38        laliga 2024-2025
       real_valladolid_cf  -0.378   -0.506         38        laliga 2024-2025
                 elche_cf   0.043   -0.074         38        laliga 2025-2026
               levante_ud   0.016   -0.105         38        laliga 2025-2026
              real_oviedo  -0.390   -0.090         38        laliga 2025-2026
                      avs  -0.347   -0.233         34 primeira_liga 2024-2025
              cd_nacional  -0.168   -0.071         34 primeira_liga 2024-2025
           cd_santa_clara  -0.124    0.277         34 primeira_liga 2024-2025
               cd_tondela  -0.273   -0.124         34 primeira_liga 2025-2026
               fc_alverca  -0.127   -0.071         34 primeira_liga 2025-2026

Mean promoted rating:
attack    -0.179
defense   -0.096
dtype: 

### Point-in-time ratings

For each matchday, fit using only FINISHED matches played before the first
kick-off of that matchday. Promoted teams get a prior built only from
promoted teams of earlier seasons (expanding prior, no lookahead).
In-memory only: nothing is written yet.

In [8]:
def promoted_prior(promoted, season):
    past = promoted[promoted["season"] < season]
    if past.empty:
        return None
    return (past["attack"].mean(), past["defense"].mean())


def build_ratings(df, league, promoted, half_life_days=365, reg=5.0):
    finished = df[(df["status"] == "FINISHED") & df["home_score"].notna()]
    last_played = finished["match_date"].max()
    seasons = sorted(df["season"].unique())
    out = []

    for prev, cur in zip([None] + seasons[:-1], seasons):
        cur_df = df[df["season"] == cur]
        season_teams = sorted(set(cur_df["home_team_key"]) | set(cur_df["away_team_key"]))

        new = set()
        if prev is not None:
            prev_df = df[df["season"] == prev]
            new = set(season_teams) - set(prev_df["home_team_key"]) - set(prev_df["away_team_key"])

        pr = promoted_prior(promoted, cur)
        prior = {t: pr for t in new} if pr else None

        # first kick-off per matchday; keep started matchdays + the next upcoming one
        starts = cur_df.groupby("matchday")["match_date"].min().sort_index()
        keep = starts[starts <= last_played]
        upcoming = starts[starts > last_played]
        if len(upcoming):
            keep = pd.concat([keep, upcoming.iloc[:1]])

        for md, cutoff in keep.items():
            train = finished[finished["match_date"] < cutoff]
            teams = sorted(set(season_teams) | set(train["home_team_key"]) | set(train["away_team_key"]))
            r, p, res = fit_dixon_coles(train, teams, cutoff, half_life_days, reg, prior)

            r = r[r["team_key"].isin(season_teams)].copy()
            r["league"], r["season"], r["matchday"] = league, cur, md
            r["as_of"] = cutoff
            r["prior_applied"] = r["team_key"].isin(new) if prior else False
            r["mu"], r["gamma"], r["rho"] = p["mu"], p["gamma"], p["rho"]
            r["converged"] = bool(res.success)
            r["n_train"] = len(train)
            r["is_warmup"] = len(train) < 100
            out.append(r)

    return pd.concat(out, ignore_index=True)


t0 = time.time()
ratings_pit = {lg: build_ratings(silver[lg], lg, promoted) for lg in LEAGUES}
print(f"Done in {time.time()-t0:.0f}s\n")

for lg, r in ratings_pit.items():
    print(f"=== {lg} === shape: {r.shape} | not converged: {int((~r['converged']).sum())}")
    print(r.groupby("season").agg(matchdays=("matchday", "nunique"),
                                  prior_rows=("prior_applied", "sum")))
    print()

# Sanity: promoted teams of 2026-27, latest snapshot
r = ratings_pit["laliga"]
cur = r[r["season"] == "2026-2027"]
last = cur[cur["matchday"] == cur["matchday"].max()]
print(last[last["team_key"].isin(["rc_deportivo", "real_racing_club", "málaga_cf"])]
      [["team_key", "attack", "defense", "n_matches", "prior_applied"]].round(3))

Done in 23s

=== laliga === shape: (2440, 15) | not converged: 0
           matchdays  prior_rows
season                          
2023-2024         38           0
2024-2025         38           0
2025-2026         38         114
2026-2027          8          24

=== primeira_liga === shape: (1980, 15) | not converged: 0
           matchdays  prior_rows
season                          
2023-2024         34           0
2024-2025         34           0
2025-2026         34          68
2026-2027          8          16

              team_key  attack  defense  n_matches  prior_applied
2428         málaga_cf  -0.416   -0.097          7           True
2431      rc_deportivo  -0.070   -0.011          7           True
2435  real_racing_club   0.027   -0.460          7           True


In [9]:
r = ratings_pit["laliga"]
chk = r[(r["season"] == "2023-2024") & (r["matchday"].isin([1, 2, 3, 5, 10, 20]))]
print(chk.groupby("matchday")[["n_train", "mu", "gamma", "rho", "is_warmup"]].first().round(3))

          n_train     mu  gamma    rho  is_warmup
matchday                                         
1               0  0.250  0.250 -0.050       True
2              10 -0.019  0.123 -0.128       True
3              20  0.043  0.118 -0.107       True
5              39  0.253  0.078 -0.056       True
10             89  0.223  0.235 -0.124       True
20            190  0.101  0.243 -0.156      False


### Write team_ratings to Gold

Validate first, then write. Already-stored matchdays are frozen
(never overwritten). Only the provisional next matchday is replaced.

In [10]:
COLS = [f.name for f in TEAM_RATINGS_SCHEMA]
KEY = ["league", "season", "matchday", "team_key"]

final = {}
for lg, r in ratings_pit.items():
    s = silver[lg]
    last_played = s.loc[(s["status"] == "FINISHED") & s["home_score"].notna(), "match_date"].max()
    r = r.copy()
    r["is_provisional"] = r["as_of"] > last_played
    final[lg] = r[COLS]

    print(f"=== {lg} ===")
    print("rows:", len(r), "| duplicate keys:", int(r.duplicated(KEY).sum()))
    print("nulls:", int(r[COLS].isna().sum().sum()))
    print("provisional matchdays:", sorted(r.loc[r["is_provisional"], "matchday"].unique()))
    print("warmup rows:", int(r["is_warmup"].sum()))
    print()

=== laliga ===
rows: 2440 | duplicate keys: 0
nulls: 0
provisional matchdays: [np.int64(8)]
warmup rows: 220

=== primeira_liga ===
rows: 1980 | duplicate keys: 0
nulls: 0
provisional matchdays: [np.int64(8)]
warmup rows: 216



In [11]:
def write_ratings(df, lake_root):
    for (lg, season), part in df.groupby(["league", "season"]):
        path = lake_root / "gold" / lg / f"season={season}" / "team_ratings.parquet"
        if path.exists():
            old = read_partition(path).to_pandas().drop(columns=["_ingested_at"], errors="ignore")
            frozen = old[~old["is_provisional"]]
            new = part[~part["matchday"].isin(frozen["matchday"])]
            part = pd.concat([frozen, new], ignore_index=True)
        part = part.sort_values(["matchday", "team_key"])
        table = pa.Table.from_pandas(part[COLS], schema=TEAM_RATINGS_SCHEMA, preserve_index=False)
        write_partition(table, path)
        print(f"{lg} {season}: {len(part)} rows")

for lg in LEAGUES:
    write_ratings(final[lg], LAKE_ROOT)

laliga 2023-2024: 760 rows
laliga 2024-2025: 760 rows
laliga 2025-2026: 760 rows
laliga 2026-2027: 160 rows
primeira_liga 2023-2024: 612 rows
primeira_liga 2024-2025: 612 rows
primeira_liga 2025-2026: 612 rows
primeira_liga 2026-2027: 144 rows


In [12]:
for lg in LEAGUES:
    g = read_league_all_seasons(LAKE_ROOT, layer="gold", league=lg,
                                filename="team_ratings.parquet").to_pandas()
    print(lg, g.shape, "| duplicate keys:", int(g.duplicated(["season", "matchday", "team_key"]).sum()))

laliga (2440, 17) | duplicate keys: 0
primeira_liga (1980, 17) | duplicate keys: 0
